# The Perceptron (Rosenblatt, 1958): Implementation / 구현

이 노트북은 Rosenblatt의 1958년 perceptron 논문의 핵심 개념을 구현합니다.
S-points → A-units → R-units 3층 구조를 NumPy로 직접 구축하고,
α/β/γ 학습 시스템을 비교하며, statistical separability의 조건을 실험적으로 검증합니다.

This notebook implements key concepts from Rosenblatt's 1958 perceptron paper.
We build the S → A → R three-layer architecture from scratch in NumPy,
compare α/β/γ learning systems, and experimentally verify statistical separability conditions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Optional
from scipy.stats import norm

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## Part 1: Rosenblatt's Perceptron from Scratch / 직접 구현

### Perceptron 구조 / Architecture

Rosenblatt의 perceptron은 3개의 층으로 구성됩니다:

1. **S-points** (감각 유닛): 입력 자극을 받는 "망막" — 이진 값 (0 또는 1)
2. **A-units** (연상 유닛): S-points로부터 무작위 연결을 받아 활성화 여부 결정
   - 각 A-unit은 $x$개의 흥분성(excitatory) + $y$개의 억제성(inhibitory) 연결
   - 활성화 조건: $\sum e_j - \sum i_k \geq \theta$
3. **R-units** (반응 유닛): A-units의 value-weighted 합으로 분류 결정

학습은 A-unit의 **value** ($V$)를 강화(reinforcement)에 따라 조정하는 것입니다.

The perceptron has three layers: S-points (sensory retina), A-units (association cells),
and R-units (response units). Learning adjusts A-unit values based on reinforcement.

In [ ]:
class RosenblattPerceptron:
    """Rosenblatt's perceptron (1958) with S-points, A-units, and R-units.

    Implements the three-layer architecture with random connectivity,
    threshold-based A-unit activation, and value-based learning.

    Attributes:
        n_s: Number of sensory units (S-points).
        n_a: Number of association units (A-units).
        n_r: Number of response units (R-units).
        x: Number of excitatory connections per A-unit.
        y: Number of inhibitory connections per A-unit.
        theta: Activation threshold for A-units.
        system_type: Learning system type ('alpha', 'beta', 'gamma').
    """

    def __init__(
        self,
        n_s: int = 100,
        n_a: int = 500,
        n_r: int = 2,
        x: int = 10,
        y: int = 5,
        theta: int = 3,
        system_type: str = 'alpha'
    ):
        self.n_s = n_s
        self.n_a = n_a
        self.n_r = n_r
        self.x = x
        self.y = y
        self.theta = theta
        self.system_type = system_type

        # Random connections: S-points -> A-units
        # Each A-unit gets x excitatory + y inhibitory connections
        self.excitatory_connections = np.zeros((n_a, n_s), dtype=bool)
        self.inhibitory_connections = np.zeros((n_a, n_s), dtype=bool)

        for i in range(n_a):
            # Randomly select x excitatory origin points
            exc_indices = np.random.choice(n_s, size=x, replace=False)
            self.excitatory_connections[i, exc_indices] = True

            # Randomly select y inhibitory origin points (from remaining)
            remaining = np.setdiff1d(np.arange(n_s), exc_indices)
            if len(remaining) >= y:
                inh_indices = np.random.choice(remaining, size=y, replace=False)
            else:
                inh_indices = remaining
            self.inhibitory_connections[i, inh_indices] = True

        # Random connections: A-units -> R-units (source-sets)
        # Each A-unit is assigned to one R-unit's source-set
        self.source_sets = np.random.randint(0, n_r, size=n_a)

        # Values (weights) for each A-unit — the learnable parameter
        self.values = np.ones(n_a, dtype=float)

    def activate_a_units(self, stimulus: np.ndarray) -> np.ndarray:
        """Compute A-unit activations for a given stimulus.

        Args:
            stimulus: Binary array of shape (n_s,) representing the input.

        Returns:
            Boolean array of shape (n_a,) indicating which A-units fire.
        """
        # Excitatory input: count active excitatory connections
        e = self.excitatory_connections @ stimulus  # shape: (n_a,)
        # Inhibitory input: count active inhibitory connections
        i = self.inhibitory_connections @ stimulus  # shape: (n_a,)
        # A-unit fires if e - i >= theta
        return (e - i) >= self.theta

    def compute_response(self, a_activations: np.ndarray) -> int:
        """Determine the dominant response (winner-take-all).

        Uses mean-discriminating (mu-system): response with highest
        mean value among its active source-set units wins.

        Args:
            a_activations: Boolean array of A-unit activations.

        Returns:
            Index of the winning response unit.
        """
        response_scores = np.zeros(self.n_r)
        for r in range(self.n_r):
            mask = (self.source_sets == r) & a_activations
            active_count = mask.sum()
            if active_count > 0:
                response_scores[r] = self.values[mask].mean()
        return int(np.argmax(response_scores))

    def reinforce(self, a_activations: np.ndarray, correct_response: int) -> None:
        """Update A-unit values based on reinforcement.

        Implements alpha, beta, or gamma system learning rules.

        Args:
            a_activations: Boolean array of A-unit activations.
            correct_response: The index of the correct response.
        """
        # Active units in the correct response's source-set
        active_in_source = (self.source_sets == correct_response) & a_activations
        n_ar = active_in_source.sum()

        if n_ar == 0:
            return

        if self.system_type == 'alpha':
            # Active units gain +1 value
            self.values[active_in_source] += 1.0

        elif self.system_type == 'beta':
            # Constant total gain K, distributed among active units
            K = 10.0
            self.values[active_in_source] += K / n_ar

        elif self.system_type == 'gamma':
            # Parasitic gain: active units gain from inactive units
            source_mask = self.source_sets == correct_response
            inactive_in_source = source_mask & (~a_activations)
            n_inactive = inactive_in_source.sum()

            if n_inactive > 0:
                # Active gain
                self.values[active_in_source] += 1.0
                # Inactive lose proportionally (total value conserved)
                loss_per_unit = float(n_ar) / float(n_inactive)
                self.values[inactive_in_source] -= loss_per_unit

    def predict(self, stimulus: np.ndarray) -> int:
        """Predict the response for a stimulus.

        Args:
            stimulus: Binary array of shape (n_s,).

        Returns:
            Predicted response index.
        """
        a_act = self.activate_a_units(stimulus)
        return self.compute_response(a_act)

    def train_step(self, stimulus: np.ndarray, label: int) -> bool:
        """One training step: present stimulus, force correct response, reinforce.

        Args:
            stimulus: Binary array of shape (n_s,).
            label: Correct response index.

        Returns:
            Whether the prediction was correct before reinforcement.
        """
        a_act = self.activate_a_units(stimulus)
        prediction = self.compute_response(a_act)
        self.reinforce(a_act, label)
        return prediction == label


# Quick test
p = RosenblattPerceptron(n_s=100, n_a=500, n_r=2, x=10, y=5, theta=3)
test_stimulus = np.random.binomial(1, 0.3, size=100)
a_act = p.activate_a_units(test_stimulus)
print(f"Stimulus: {test_stimulus.sum()} active S-points")
print(f"A-units activated: {a_act.sum()} / {p.n_a}")
print(f"Response: R{p.compute_response(a_act)}")

## Part 2: Visualizing $P_a$ — Activation Probability / 활성화 확률 시각화

### 논문의 핵심 변수 $P_a$ / The Key Variable $P_a$

$P_a$는 주어진 자극에 의해 임의의 A-unit이 활성화될 확률입니다.
이것은 자극 크기($R$), 흥분/억제 연결 수($x$, $y$), 임계값($\theta$)에 의존합니다.

$$P_a = \sum_{e,i} \binom{x}{e} R^e(1-R)^{x-e} \binom{y}{i} R^i(1-R)^{y-i} \cdot \mathbb{1}[e-i \geq \theta]$$

아래에서 $P_a$를 이론적으로 계산하고, 시뮬레이션 결과와 비교합니다.

$P_a$ is the probability that a random A-unit fires for a given stimulus.
We compute it analytically and compare with simulation.

In [ ]:
from scipy.special import comb


def compute_pa_theoretical(R: float, x: int, y: int, theta: int) -> float:
    """Compute P_a analytically using Rosenblatt's formula.

    Args:
        R: Proportion of S-points illuminated by stimulus.
        x: Number of excitatory connections per A-unit.
        y: Number of inhibitory connections per A-unit.
        theta: Activation threshold.

    Returns:
        P_a: probability of A-unit activation.
    """
    pa = 0.0
    for e in range(x + 1):
        for i in range(y + 1):
            if e - i >= theta:
                p_e = comb(x, e, exact=True) * (R ** e) * ((1 - R) ** (x - e))
                p_i = comb(y, i, exact=True) * (R ** i) * ((1 - R) ** (y - i))
                pa += p_e * p_i
    return pa


def compute_pa_simulation(
    R: float, x: int, y: int, theta: int, n_trials: int = 10000
) -> float:
    """Estimate P_a via Monte Carlo simulation.

    Args:
        R: Proportion of S-points illuminated.
        x: Number of excitatory connections.
        y: Number of inhibitory connections.
        theta: Activation threshold.
        n_trials: Number of simulation trials.

    Returns:
        Estimated P_a.
    """
    excit = np.random.binomial(x, R, size=n_trials)
    inhib = np.random.binomial(y, R, size=n_trials)
    return np.mean((excit - inhib) >= theta)


# --- Fig. 4 reproduction: P_a as function of R ---
R_values = np.linspace(0.01, 0.5, 50)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Effect of inhibitory-excitatory mixture, theta=1
configs_a = [
    (10, 0, 1, 'x=10, y=0'),
    (8, 2, 1, 'x=8, y=2'),
    (5, 5, 1, 'x=5, y=5'),
]
for x, y, theta, label in configs_a:
    pa_vals = [compute_pa_theoretical(R, x, y, theta) for R in R_values]
    axes[0].plot(R_values, pa_vals, label=label, linewidth=2)
axes[0].set_title('(a) Effect of inhibitory/excitatory mixture\n(θ=1)')
axes[0].set_xlabel('R (proportion of S-points illuminated)')
axes[0].set_ylabel('$P_a$')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# (b) Variation with theta, x=10, y=0
for theta in [1, 2, 3, 5]:
    pa_vals = [compute_pa_theoretical(R, 10, 0, theta) for R in R_values]
    axes[1].plot(R_values, pa_vals, label=f'θ={theta}', linewidth=2)
axes[1].set_title('(b) Variation with θ\n(x=10, y=0)')
axes[1].set_xlabel('R')
axes[1].set_ylabel('$P_a$')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# (c) Balanced mixture, flattened curves
configs_c = [
    (10, 0, 3, 'x=10, y=0, θ=3'),
    (5, 5, 1, 'x=5, y=5, θ=1'),
    (5, 5, 2, 'x=5, y=5, θ=2'),
]
for x, y, theta, label in configs_c:
    pa_vals = [compute_pa_theoretical(R, x, y, theta) for R in R_values]
    axes[2].plot(R_values, pa_vals, label=label, linewidth=2)
axes[2].set_title('(c) Balanced mixture flattens curves\n(excitation ≈ inhibition)')
axes[2].set_xlabel('R')
axes[2].set_ylabel('$P_a$')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle("Fig. 4 Reproduction: $P_a$ as function of retinal area illuminated",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Verify theory vs simulation
R_test = 0.3
pa_theory = compute_pa_theoretical(R_test, 10, 5, 3)
pa_sim = compute_pa_simulation(R_test, 10, 5, 3)
print(f"P_a (R={R_test}, x=10, y=5, θ=3):")
print(f"  Theory:     {pa_theory:.4f}")
print(f"  Simulation: {pa_sim:.4f}")

## Part 3: $P_c$ — Co-activation Probability / 공동 활성화 확률

### 두 자극의 "신경적 유사성" / Neural Similarity Between Stimuli

$P_c$는 자극 $S_1$에 반응한 A-unit이 $S_2$에도 반응할 조건부 확률입니다.
이것이 perceptron이 두 자극을 "유사하게" 취급하는 정도를 결정합니다.

- $P_c \to 1$: 두 자극이 거의 동일하게 처리됨 (일반화)
- $P_c \to 0$: 두 자극이 완전히 다르게 처리됨 (판별)

$P_c$ is the conditional probability that an A-unit responding to $S_1$
also responds to $S_2$. It measures neural similarity between stimuli.

In [ ]:
def estimate_pc_simulation(
    n_s: int, x: int, y: int, theta: int,
    R: float, overlap: float, n_units: int = 5000
) -> float:
    """Estimate P_c via simulation.

    Creates two stimuli with a given overlap and measures the conditional
    probability that A-units responding to S1 also respond to S2.

    Args:
        n_s: Number of sensory units.
        x: Number of excitatory connections per A-unit.
        y: Number of inhibitory connections per A-unit.
        theta: Activation threshold.
        R: Proportion of S-points active in each stimulus.
        overlap: Proportion of active S-points shared between S1 and S2
                 (relative to the active set of S1).
        n_units: Number of A-units to simulate.

    Returns:
        Estimated P_c.
    """
    n_active = int(R * n_s)
    n_shared = int(overlap * n_active)

    # Create S1 and S2
    s1 = np.zeros(n_s, dtype=bool)
    active_indices = np.random.choice(n_s, size=n_active, replace=False)
    s1[active_indices] = True

    s2 = np.zeros(n_s, dtype=bool)
    # Shared points
    shared = active_indices[:n_shared]
    s2[shared] = True
    # Unique to S2
    remaining = np.setdiff1d(np.arange(n_s), active_indices)
    n_unique = n_active - n_shared
    if n_unique > 0 and len(remaining) >= n_unique:
        unique_s2 = np.random.choice(remaining, size=n_unique, replace=False)
        s2[unique_s2] = True

    responds_s1 = 0
    responds_both = 0

    for _ in range(n_units):
        # Random connections for this A-unit
        exc = np.random.choice(n_s, size=x, replace=False)
        rem = np.setdiff1d(np.arange(n_s), exc)
        inh = np.random.choice(rem, size=min(y, len(rem)), replace=False)

        # Check activation for S1
        e1 = s1[exc].sum()
        i1 = s1[inh].sum()
        fires_s1 = (e1 - i1) >= theta

        if fires_s1:
            responds_s1 += 1
            # Check activation for S2
            e2 = s2[exc].sum()
            i2 = s2[inh].sum()
            fires_s2 = (e2 - i2) >= theta
            if fires_s2:
                responds_both += 1

    return responds_both / responds_s1 if responds_s1 > 0 else 0.0


# --- Fig. 6 reproduction: P_c as function of overlap ---
overlap_values = np.linspace(0, 1, 20)

fig, ax = plt.subplots(figsize=(10, 6))

configs = [
    (0.5, 10, 0, 1, 'R=0.5, θ=1'),
    (0.5, 10, 0, 3, 'R=0.5, θ=3'),
    (0.2, 10, 0, 1, 'R=0.2, θ=1'),
    (0.2, 10, 0, 3, 'R=0.2, θ=3'),
]

for R, x, y, theta, label in configs:
    pc_vals = []
    for ov in overlap_values:
        pc = estimate_pc_simulation(200, x, y, theta, R, ov, n_units=3000)
        pc_vals.append(pc)
    style = '-' if R == 0.5 else '--'
    ax.plot(overlap_values, pc_vals, style, label=label, linewidth=2)

ax.set_xlabel('C (proportion of overlap between stimuli)')
ax.set_ylabel('$P_c$')
ax.set_title('Fig. 6 Reproduction: $P_c$ as function of stimulus overlap\n'
             '(solid: R=0.5, dashed: R=0.2)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Part 4: Learning in the Ideal Environment / 이상적 환경에서의 학습

### 이상적 환경이란? / What is an Ideal Environment?

"이상적 환경"은 자극이 무작위 점들의 집합인 경우입니다 — 고유한 클래스 구조가 없습니다.
자극을 임의로 두 그룹으로 나누고 각각 $R_1$, $R_2$에 할당합니다.

이 환경에서:
- **$P_r$ (recall)**: 이전에 본 자극을 기억 → 가능 (하지만 학습량 많으면 성능 저하)
- **$P_g$ (generalization)**: 새 자극 분류 → **불가능** ($c_1 = 0$이므로 $Z = 0$)

In the ideal environment, stimuli are random collections of active S-points with
no class structure. Recall is possible but generalization is not.

In [ ]:
def run_ideal_environment_experiment(
    system_type: str,
    n_stimuli_per_class: int = 50,
    n_epochs: int = 20,
    n_s: int = 100,
    n_a: int = 500,
    R: float = 0.3
) -> Tuple[List[float], List[float]]:
    """Run a learning experiment in an ideal (random) environment.

    Args:
        system_type: 'alpha', 'beta', or 'gamma'.
        n_stimuli_per_class: Number of stimuli assigned to each response.
        n_epochs: Number of training epochs.
        n_s: Number of sensory units.
        n_a: Number of association units.
        R: Proportion of active S-points per stimulus.

    Returns:
        Tuple of (recall_accuracies, generalization_accuracies) per epoch.
    """
    perc = RosenblattPerceptron(
        n_s=n_s, n_a=n_a, n_r=2, x=10, y=5, theta=3,
        system_type=system_type
    )

    # Create random stimuli (ideal environment)
    train_stimuli = [np.random.binomial(1, R, size=n_s)
                     for _ in range(2 * n_stimuli_per_class)]
    train_labels = [0] * n_stimuli_per_class + [1] * n_stimuli_per_class

    # Test stimuli: new random stimuli from same "classes"
    test_stimuli = [np.random.binomial(1, R, size=n_s) for _ in range(100)]
    test_labels = [0] * 50 + [1] * 50

    recall_accs = []
    gen_accs = []

    for epoch in range(n_epochs):
        # Train
        indices = np.random.permutation(len(train_stimuli))
        for idx in indices:
            perc.train_step(train_stimuli[idx], train_labels[idx])

        # Evaluate recall (same stimuli)
        correct_recall = sum(
            perc.predict(train_stimuli[i]) == train_labels[i]
            for i in range(len(train_stimuli))
        )
        recall_accs.append(correct_recall / len(train_stimuli))

        # Evaluate generalization (new stimuli)
        correct_gen = sum(
            perc.predict(test_stimuli[i]) == test_labels[i]
            for i in range(len(test_stimuli))
        )
        gen_accs.append(correct_gen / len(test_stimuli))

    return recall_accs, gen_accs


# Run for all three systems
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, sys_type in enumerate(['alpha', 'beta', 'gamma']):
    recall, gen = run_ideal_environment_experiment(sys_type, n_stimuli_per_class=30)
    epochs = range(1, len(recall) + 1)
    axes[idx].plot(epochs, recall, 'b-o', label='$P_r$ (recall)', markersize=4)
    axes[idx].plot(epochs, gen, 'r-s', label='$P_g$ (generalization)', markersize=4)
    axes[idx].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='chance')
    axes[idx].set_title(f'{sys_type.capitalize()}-system')
    axes[idx].set_xlabel('Epoch')
    axes[idx].set_ylabel('Accuracy')
    axes[idx].set_ylim(0.3, 1.0)
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Ideal Environment: Recall vs Generalization\n'
             '(random stimuli, no class structure — $P_g$ should stay at chance)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 5: Learning in Differentiated Environment / 차별화된 환경에서의 학습

### 차별화된 환경 — 진정한 학습 / The Differentiated Environment

이제 자극에 **구조**를 부여합니다. 클래스 1의 자극은 망막의 상반부에,
클래스 2는 하반부에 주로 분포합니다 (일부 겹침 허용).

이 환경에서는 $c_1 \neq 0$이므로:
- $P_r$이 향상됨 (당연)
- **$P_g$도 향상됨** (핵심!)
- 충분한 학습 후 $P_r \to P_g$ (같은 점근값에 수렴)

일반화 조건: $P_{c12} < P_a < P_{c11}$

Now stimuli have structure: Class 1 activates mostly the upper retina,
Class 2 the lower. Both recall AND generalization improve, and they
converge to the same asymptote.

In [ ]:
def create_structured_stimulus(
    n_s: int, label: int, R: float = 0.3, structure_strength: float = 0.7
) -> np.ndarray:
    """Create a stimulus with class-dependent spatial structure.

    Class 0 stimuli are biased toward the first half of the retina.
    Class 1 stimuli are biased toward the second half.

    Args:
        n_s: Number of sensory units.
        label: Class label (0 or 1).
        R: Overall proportion of active S-points.
        structure_strength: How strongly the class determines spatial location
                          (1.0 = no overlap, 0.5 = random).

    Returns:
        Binary stimulus array.
    """
    stimulus = np.zeros(n_s, dtype=int)
    n_active = int(R * n_s)
    half = n_s // 2

    # Probabilities for each S-point
    probs = np.ones(n_s) / n_s
    if label == 0:
        probs[:half] *= structure_strength * 2
        probs[half:] *= (1 - structure_strength) * 2
    else:
        probs[:half] *= (1 - structure_strength) * 2
        probs[half:] *= structure_strength * 2
    probs /= probs.sum()

    active = np.random.choice(n_s, size=n_active, replace=False, p=probs)
    stimulus[active] = 1
    return stimulus


def run_differentiated_experiment(
    system_type: str,
    n_train: int = 50,
    n_epochs: int = 30,
    structure_strength: float = 0.7
) -> Tuple[List[float], List[float]]:
    """Run learning experiment in a differentiated environment.

    Args:
        system_type: 'alpha', 'beta', or 'gamma'.
        n_train: Number of training stimuli per class.
        n_epochs: Number of training epochs.
        structure_strength: Degree of class structure.

    Returns:
        Tuple of (recall_accuracies, generalization_accuracies).
    """
    n_s = 100
    perc = RosenblattPerceptron(
        n_s=n_s, n_a=500, n_r=2, x=10, y=5, theta=3,
        system_type=system_type
    )

    # Create structured training stimuli
    train_stimuli = (
        [create_structured_stimulus(n_s, 0, structure_strength=structure_strength)
         for _ in range(n_train)] +
        [create_structured_stimulus(n_s, 1, structure_strength=structure_strength)
         for _ in range(n_train)]
    )
    train_labels = [0] * n_train + [1] * n_train

    recall_accs = []
    gen_accs = []

    for epoch in range(n_epochs):
        # Train
        indices = np.random.permutation(len(train_stimuli))
        for idx in indices:
            perc.train_step(train_stimuli[idx], train_labels[idx])

        # Recall: test on training stimuli
        correct_r = sum(
            perc.predict(train_stimuli[i]) == train_labels[i]
            for i in range(len(train_stimuli))
        )
        recall_accs.append(correct_r / len(train_stimuli))

        # Generalization: test on NEW stimuli from same classes
        n_test = 100
        test_stimuli = (
            [create_structured_stimulus(n_s, 0,
                                       structure_strength=structure_strength)
             for _ in range(n_test // 2)] +
            [create_structured_stimulus(n_s, 1,
                                       structure_strength=structure_strength)
             for _ in range(n_test // 2)]
        )
        test_labels = [0] * (n_test // 2) + [1] * (n_test // 2)
        correct_g = sum(
            perc.predict(test_stimuli[i]) == test_labels[i]
            for i in range(n_test)
        )
        gen_accs.append(correct_g / n_test)

    return recall_accs, gen_accs


# Compare all three systems in differentiated environment
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, sys_type in enumerate(['alpha', 'beta', 'gamma']):
    recall, gen = run_differentiated_experiment(sys_type)
    epochs = range(1, len(recall) + 1)
    axes[idx].plot(epochs, recall, 'b-o', label='$P_r$ (recall)', markersize=4)
    axes[idx].plot(epochs, gen, 'r-s', label='$P_g$ (generalization)', markersize=4)
    axes[idx].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='chance')
    axes[idx].set_title(f'{sys_type.capitalize()}-system')
    axes[idx].set_xlabel('Epoch')
    axes[idx].set_ylabel('Accuracy')
    axes[idx].set_ylim(0.3, 1.0)
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Differentiated Environment: $P_r$ and $P_g$ both improve!\n'
             '(structured stimuli — both should converge to same asymptote)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 6: Comparing α, β, γ Systems / 세 시스템 비교

### 성능 비교 — γ-system의 우수성 / Performance Comparison

Rosenblatt은 γ-system이 가장 우수하다고 예측했습니다.
이유: 총 value가 보존되어 잡음 축적이 없고, 경쟁적 학습이 관련 연결을 강화합니다.
아래에서 세 시스템의 value 분포 변화와 최종 성능을 비교합니다.

Rosenblatt predicted that the γ-system would outperform α and β.
Reason: total value is conserved (no noise accumulation), and competitive learning strengthens relevant connections.
Below we compare value distribution evolution and final performance across the three systems.

In [ ]:
def analyze_value_dynamics(
    system_type: str, n_epochs: int = 20
) -> Tuple[List[float], List[float], np.ndarray]:
    """Analyze how A-unit values evolve during training.

    Args:
        system_type: 'alpha', 'beta', or 'gamma'.
        n_epochs: Number of training epochs.

    Returns:
        Tuple of (mean_values, total_values, final_values).
    """
    n_s = 100
    perc = RosenblattPerceptron(
        n_s=n_s, n_a=500, n_r=2, x=10, y=5, theta=3,
        system_type=system_type
    )

    train_data = (
        [create_structured_stimulus(n_s, 0) for _ in range(30)] +
        [create_structured_stimulus(n_s, 1) for _ in range(30)]
    )
    labels = [0] * 30 + [1] * 30

    mean_vals = [perc.values.mean()]
    total_vals = [perc.values.sum()]

    for epoch in range(n_epochs):
        indices = np.random.permutation(len(train_data))
        for idx in indices:
            perc.train_step(train_data[idx], labels[idx])
        mean_vals.append(perc.values.mean())
        total_vals.append(perc.values.sum())

    return mean_vals, total_vals, perc.values.copy()


fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for idx, sys_type in enumerate(['alpha', 'beta', 'gamma']):
    means, totals, final_vals = analyze_value_dynamics(sys_type)

    # Top row: total value over time
    axes[0, idx].plot(totals, 'k-', linewidth=2)
    axes[0, idx].set_title(f'{sys_type.capitalize()}: Total System Value')
    axes[0, idx].set_xlabel('Epoch')
    axes[0, idx].set_ylabel('$\\sum V_i$')
    axes[0, idx].grid(True, alpha=0.3)

    # Bottom row: final value distribution
    axes[1, idx].hist(final_vals, bins=30, alpha=0.7, color='steelblue',
                      edgecolor='black')
    axes[1, idx].set_title(f'{sys_type.capitalize()}: Final Value Distribution')
    axes[1, idx].set_xlabel('Value ($V$)')
    axes[1, idx].set_ylabel('Count')
    axes[1, idx].axvline(final_vals.mean(), color='red', linestyle='--',
                         label=f'mean={final_vals.mean():.1f}')
    axes[1, idx].legend()
    axes[1, idx].grid(True, alpha=0.3)

plt.suptitle('Value Dynamics: α grows unbounded, β grows faster, γ is conserved',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 7: The Generalization Condition / 일반화 조건 검증

### $P_{c12} < P_a < P_{c11}$ — 일반화가 가능한 조건

Rosenblatt의 핵심 결과: 같은 클래스 내 유사성($P_{c11}$)이 $P_a$보다 크고,
다른 클래스 간 유사성($P_{c12}$)이 $P_a$보다 작을 때 일반화가 가능합니다.

이것은 현대의 "intra-class variance < inter-class variance" 조건과 동일합니다.
아래에서 구조의 강도를 변화시키며 이 조건과 일반화 성능의 관계를 확인합니다.

We verify that generalization accuracy tracks the condition $P_{c12} < P_a < P_{c11}$
by varying the structure strength of the stimulus environment.

In [ ]:
def measure_pc_for_classes(
    n_s: int, x: int, y: int, theta: int,
    structure_strength: float, n_pairs: int = 500
) -> Tuple[float, float, float]:
    """Measure P_c11 (within-class) and P_c12 (between-class) empirically.

    Args:
        n_s: Number of sensory units.
        x: Excitatory connections.
        y: Inhibitory connections.
        theta: Threshold.
        structure_strength: Degree of class structure.
        n_pairs: Number of stimulus pairs to evaluate.

    Returns:
        Tuple of (P_a, P_c11, P_c12).
    """
    # Generate stimuli
    stim_class0 = [create_structured_stimulus(n_s, 0,
                   structure_strength=structure_strength) for _ in range(50)]
    stim_class1 = [create_structured_stimulus(n_s, 1,
                   structure_strength=structure_strength) for _ in range(50)]

    # Create random A-units and measure co-activations
    pa_total, pc11_total, pc12_total = 0, 0, 0
    pa_count, pc11_count, pc12_count = 0, 0, 0

    for _ in range(n_pairs):
        # Random A-unit
        exc = np.random.choice(n_s, size=x, replace=False)
        rem = np.setdiff1d(np.arange(n_s), exc)
        inh = np.random.choice(rem, size=min(y, len(rem)), replace=False)

        # Pick random stimuli
        s0a, s0b = [stim_class0[i] for i in np.random.choice(50, 2, replace=False)]
        s1a = stim_class1[np.random.randint(50)]

        def fires(s: np.ndarray) -> bool:
            return int(s[exc].sum() - s[inh].sum()) >= theta

        f0a = fires(s0a)
        f0b = fires(s0b)
        f1a = fires(s1a)

        pa_total += f0a
        pa_count += 1

        # P_c11: within-class (both from class 0)
        if f0a:
            pc11_total += f0b
            pc11_count += 1

        # P_c12: between-class (class 0 vs class 1)
        if f0a:
            pc12_total += f1a
            pc12_count += 1

    pa = pa_total / pa_count if pa_count > 0 else 0
    pc11 = pc11_total / pc11_count if pc11_count > 0 else 0
    pc12 = pc12_total / pc12_count if pc12_count > 0 else 0

    return pa, pc11, pc12


# Vary structure strength and measure condition + generalization
strengths = np.linspace(0.5, 0.95, 10)
pa_vals, pc11_vals, pc12_vals, gen_vals = [], [], [], []

for strength in strengths:
    pa, pc11, pc12 = measure_pc_for_classes(100, 10, 5, 3, strength, n_pairs=1000)
    pa_vals.append(pa)
    pc11_vals.append(pc11)
    pc12_vals.append(pc12)

    # Measure generalization performance
    _, gen_acc = run_differentiated_experiment(
        'gamma', n_train=30, n_epochs=15, structure_strength=strength
    )
    gen_vals.append(gen_acc[-1])  # Final epoch accuracy

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: P_a, P_c11, P_c12
ax1.plot(strengths, pa_vals, 'k-o', label='$P_a$', linewidth=2)
ax1.plot(strengths, pc11_vals, 'b-s', label='$P_{c11}$ (within-class)', linewidth=2)
ax1.plot(strengths, pc12_vals, 'r-^', label='$P_{c12}$ (between-class)', linewidth=2)
ax1.set_xlabel('Structure Strength')
ax1.set_ylabel('Probability')
ax1.set_title('Generalization Condition: $P_{c12} < P_a < P_{c11}$')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Generalization accuracy
ax2.plot(strengths, gen_vals, 'g-o', linewidth=2, markersize=8)
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='chance')
ax2.set_xlabel('Structure Strength')
ax2.set_ylabel('Generalization Accuracy')
ax2.set_title('$P_g$ Improves as Class Structure Increases')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Verifying the Generalization Condition',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print the condition check
print("\nCondition Check: P_c12 < P_a < P_c11")
print(f"{'Strength':>10} {'P_c12':>8} {'P_a':>8} {'P_c11':>8} {'Condition':>10} {'P_g':>8}")
for s, pa, pc11, pc12, pg in zip(strengths, pa_vals, pc11_vals, pc12_vals, gen_vals):
    met = 'YES' if pc12 < pa < pc11 else 'NO'
    print(f"{s:>10.2f} {pc12:>8.3f} {pa:>8.3f} {pc11:>8.3f} {met:>10} {pg:>8.2f}")

## Part 8: The XOR Problem — Perceptron's Fundamental Limit / XOR 문제 — 근본적 한계

### 왜 단층 Perceptron은 XOR을 풀 수 없는가? / Why Can't a Single-Layer Perceptron Solve XOR?

Rosenblatt 자신이 인정한 한계: "관계 판단과 추상화"가 불가능합니다.
가장 유명한 예가 XOR 문제입니다.

Rosenblatt himself acknowledged this limit: the perceptron cannot perform "relative judgment" or "abstraction of relationships."
The most famous example is the XOR problem:

| $x_1$ | $x_2$ | XOR |
|-------|-------|-----|
| 0     | 0     | 0   |
| 0     | 1     | 1   |
| 1     | 0     | 1   |
| 1     | 1     | 0   |

XOR은 **선형 분리 불가능**(linearly inseparable)합니다 — 2D 평면에서
하나의 직선으로 두 클래스를 나눌 수 없습니다.
이것을 시각화하고, 현대의 다층 perceptron(MLP)과 비교합니다.

XOR is **linearly inseparable** — no single straight line in 2D can separate the two classes.
We visualize this and compare with the modern multi-layer perceptron (MLP).

In [ ]:
# --- XOR: Perceptron's fundamental limit ---

# Modern single-layer perceptron (simplified)
class SimplePerceptron:
    """Modern single-layer perceptron for comparison.

    Uses the standard perceptron learning rule:
    w <- w + lr * (y - y_hat) * x
    """

    def __init__(self, n_inputs: int, lr: float = 0.1):
        self.weights = np.random.randn(n_inputs + 1) * 0.1  # +1 for bias
        self.lr = lr

    def predict(self, x: np.ndarray) -> int:
        """Binary prediction with threshold at 0."""
        x_bias = np.append(x, 1)
        return 1 if np.dot(self.weights, x_bias) >= 0 else 0

    def train(self, X: np.ndarray, y: np.ndarray, n_epochs: int = 100) -> List[float]:
        """Train and return accuracy per epoch."""
        accs = []
        for epoch in range(n_epochs):
            for xi, yi in zip(X, y):
                pred = self.predict(xi)
                x_bias = np.append(xi, 1)
                self.weights += self.lr * (yi - pred) * x_bias
            acc = np.mean([self.predict(xi) == yi for xi, yi in zip(X, y)])
            accs.append(acc)
        return accs


# Define problems
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

problems = {
    'AND': np.array([0, 0, 0, 1]),
    'OR': np.array([0, 1, 1, 1]),
    'XOR': np.array([0, 1, 1, 0]),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (name, y) in enumerate(problems.items()):
    # Train
    perc = SimplePerceptron(2, lr=0.1)
    accs = perc.train(X, y, n_epochs=50)

    # Decision boundary visualization
    ax = axes[idx]
    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200),
                         np.linspace(-0.5, 1.5, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    preds = np.array([perc.predict(p) for p in grid]).reshape(xx.shape)

    ax.contourf(xx, yy, preds, alpha=0.3, levels=[-0.5, 0.5, 1.5],
                colors=['#ff9999', '#99ff99'])
    ax.contour(xx, yy, preds, levels=[0.5], colors='black', linewidths=2)

    # Plot data points
    for i in range(4):
        color = 'green' if y[i] == 1 else 'red'
        marker = 'o' if y[i] == 1 else 's'
        ax.scatter(X[i, 0], X[i, 1], c=color, s=200, marker=marker,
                   edgecolors='black', linewidth=2, zorder=5)

    final_acc = accs[-1]
    ax.set_title(f'{name}\nAccuracy: {final_acc:.0%}\n'
                 f'{"✓ Linearly Separable" if final_acc == 1.0 else "✗ NOT Linearly Separable"}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.grid(True, alpha=0.3)

plt.suptitle("Perceptron's Fundamental Limit: XOR is NOT Linearly Separable\n"
             "(Rosenblatt acknowledged this; Minsky & Papert proved it formally in 1969)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 9: Modern Comparison — scikit-learn Perceptron / 현대 구현과 비교

### Rosenblatt → 현대 Perceptron / From Rosenblatt to Modern Implementation

Rosenblatt의 perceptron은 현대의 `sklearn.linear_model.Perceptron`과 본질적으로
같은 알고리즘입니다. 차이점:

| Rosenblatt (1958) | Modern Perceptron |
|---|---|
| S → A → R 3층 | Input → Output 2층 (A-units는 feature extraction) |
| Value ($V$) | Weight ($w$) |
| Forced reinforcement | Perceptron learning rule: $\Delta w = \eta(y - \hat{y})x$ |
| μ-system or Σ-system | Weighted sum + threshold |
| Statistical separability | Linear separability |

We compare Rosenblatt's implementation with scikit-learn's Perceptron.

In [ ]:
from sklearn.linear_model import Perceptron
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


# Generate a structured classification dataset
X_data, y_data = make_classification(
    n_samples=500, n_features=20, n_informative=10,
    n_classes=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.3, random_state=42
)

# --- sklearn Perceptron ---
sklearn_perc = Perceptron(max_iter=50, random_state=42)
sklearn_perc.fit(X_train, y_train)
sklearn_acc = accuracy_score(y_test, sklearn_perc.predict(X_test))

# --- Rosenblatt-style Perceptron ---
# Binarize inputs (Rosenblatt's perceptron uses binary S-points)
X_binary_train = (X_train > X_train.mean(axis=0)).astype(int)
X_binary_test = (X_test > X_test.mean(axis=0)).astype(int)

rosenblatt = RosenblattPerceptron(
    n_s=20, n_a=200, n_r=2, x=8, y=4, theta=2, system_type='gamma'
)

# Train Rosenblatt perceptron
rosenblatt_train_accs = []
rosenblatt_test_accs = []
for epoch in range(50):
    indices = np.random.permutation(len(X_binary_train))
    for i in indices:
        rosenblatt.train_step(X_binary_train[i], y_train[i])

    train_preds = [rosenblatt.predict(x) for x in X_binary_train]
    test_preds = [rosenblatt.predict(x) for x in X_binary_test]
    rosenblatt_train_accs.append(accuracy_score(y_train, train_preds))
    rosenblatt_test_accs.append(accuracy_score(y_test, test_preds))

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Rosenblatt learning curves
ax1.plot(rosenblatt_train_accs, 'b-', label='Train ($P_r$)', linewidth=2)
ax1.plot(rosenblatt_test_accs, 'r-', label='Test ($P_g$)', linewidth=2)
ax1.axhline(y=sklearn_acc, color='green', linestyle='--', linewidth=2,
            label=f'sklearn Perceptron: {sklearn_acc:.1%}')
ax1.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Rosenblatt Perceptron (γ-system) vs sklearn')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0.4, 1.0)

# Right: Final comparison bar chart
methods = ['Rosenblatt\n(γ-system)', 'sklearn\nPerceptron']
accs = [rosenblatt_test_accs[-1], sklearn_acc]
colors = ['steelblue', 'forestgreen']
bars = ax2.bar(methods, accs, color=colors, edgecolor='black', width=0.5)
ax2.set_ylabel('Test Accuracy')
ax2.set_title('Final Test Accuracy Comparison')
ax2.set_ylim(0, 1.0)
ax2.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
for bar, acc in zip(bars, accs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{acc:.1%}', ha='center', fontweight='bold', fontsize=14)
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('1958 vs Modern: Same Core Idea, Different Implementation',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nRosenblatt Perceptron (γ-system): {rosenblatt_test_accs[-1]:.1%}")
print(f"sklearn Perceptron:               {sklearn_acc:.1%}")
print(f"\nNote: sklearn uses continuous features and gradient-based updates,")
print(f"while Rosenblatt uses binary inputs and value-based reinforcement.")
print(f"Both are fundamentally single-layer linear classifiers.")

## Summary: Rosenblatt's Legacy / 요약: Rosenblatt의 유산

| Concept / 개념 | Rosenblatt (1958) | Modern Equivalent / 현대 동등 | Next Paper / 다음 논문 |
|---|---|---|---|
| Trainable weights | Value ($V$) of A-units | Weights ($w$) in neural networks | #6 Rumelhart — backpropagation |
| Threshold activation | $e - i \geq \theta$ | ReLU, sigmoid, etc. | #6 Rumelhart — continuous activations |
| Random connectivity | Random S→A connections | Random initialization of weights | All modern networks |
| γ-system (competitive) | Conserved total value | Weight normalization, dropout | #12 Hinton — RBMs, regularization |
| Statistical separability | $P_{c12} < P_a < P_{c11}$ | Margin, VC dimension | #8 Cortes & Vapnik — SVMs |
| Generalization = Recall | $P_r \to P_g$ asymptotically | Train/test convergence | All of statistical learning theory |
| Distributed memory | Removal degrades gracefully | Distributed representations | #5 Hopfield — content-addressable memory |
| Linear inseparability limit | Cannot abstract relationships | XOR problem | **#4 Minsky & Papert (1969)** |
| Spontaneous concepts | Unsupervised class discovery | Self-organizing maps, clustering | Kohonen (1982) |

### 다음 논문 / Next Paper

**#4 Minsky & Papert — "Perceptrons" (1969)**

Rosenblatt이 인정한 한계를 수학적으로 엄밀하게 증명합니다.
XOR뿐만 아니라 "연결성(connectedness)"과 같은 근본적으로 선형 분리 불가능한
문제들을 보이며, 신경망 연구에 10년 이상의 "겨울"을 초래합니다.